<table align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/glasslego/ml-deep-learning-study/blob/main/src/notebook/7-0_deep_learning_basic.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />구글 코랩에서 실행하기</a>
  </td>
</table>

# Chapter 6: 깊은 인공 신경망의 고질적 문제와 해결 방안

이 노트북은 딥러닝의 주요 문제점들(기울기 소실, 과적합 등)과 해결 방법들을 실습합니다.

In [ ]:
# Colab 환경 체크 및 필요한 라이브러리 설치
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Google Colab 환경에서 실행 중")
    # 나눔고딕 폰트 설치
    !sudo apt-get install -y fonts-nanum
    !sudo fc-cache -fv
    !rm ~/.cache/matplotlib -rf
else:
    print("로컬 환경에서 실행 중")

# 필요한 라이브러리 설치 및 임포트
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_circles
from sklearn.model_selection import train_test_split

# 한글 폰트 설정
plt.rc('font', family='NanumGothic')
plt.rc('axes', unicode_minus=False)  # 마이너스 기호 깨짐 방지

# 재현성을 위한 시드 설정
torch.manual_seed(42)
np.random.seed(42)

# GPU 사용 가능 여부 확인
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 6.1. 기울기 소실(Vanishing Gradient)과 과소적합(Underfitting)

### 6.1.1. ReLU (Rectified Linear Unit)

In [ ]:
# ReLU 함수 시각화
x = np.linspace(-5, 5, 100)
relu = np.maximum(0, x)
leaky_relu = np.where(x > 0, x, 0.01 * x)

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.plot(x, relu, label='ReLU', color='blue', linewidth=2)
plt.grid(True, alpha=0.3)
plt.xlabel('Input')
plt.ylabel('Output')
plt.title('ReLU Activation Function')
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(x, leaky_relu, label='Leaky ReLU', color='green', linewidth=2)
plt.grid(True, alpha=0.3)
plt.xlabel('Input')
plt.ylabel('Output')
plt.title('Leaky ReLU Activation Function')
plt.legend()

plt.subplot(1, 3, 3)
relu_grad = np.where(x > 0, 1, 0)
plt.plot(x, relu_grad, label='ReLU Gradient', color='red', linewidth=2)
plt.grid(True, alpha=0.3)
plt.xlabel('Input')
plt.ylabel('Gradient')
plt.title('ReLU Gradient')
plt.legend()

plt.tight_layout()
plt.show()

### 6.1.2. Sigmoid vs ReLU 실험 결과 분석

In [ ]:
# Sigmoid를 사용한 깊은 신경망
class DeepNetworkSigmoid(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(DeepNetworkSigmoid, self).__init__()
        self.layers = nn.ModuleList()
        
        # 첫 번째 레이어
        self.layers.append(nn.Linear(input_size, hidden_size))
        
        # 히든 레이어들
        for _ in range(num_layers - 1):
            self.layers.append(nn.Linear(hidden_size, hidden_size))
        
        # 출력 레이어
        self.output = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        for layer in self.layers:
            x = torch.sigmoid(layer(x))
        x = self.output(x)
        return x

# ReLU를 사용한 깊은 신경망
class DeepNetworkReLU(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(DeepNetworkReLU, self).__init__()
        self.layers = nn.ModuleList()
        
        # 첫 번째 레이어
        self.layers.append(nn.Linear(input_size, hidden_size))
        
        # 히든 레이어들
        for _ in range(num_layers - 1):
            self.layers.append(nn.Linear(hidden_size, hidden_size))
        
        # 출력 레이어
        self.output = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        for layer in self.layers:
            x = F.relu(layer(x))
        x = self.output(x)
        return x

# 간단한 데이터셋 생성
X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 텐서로 변환
X_train = torch.FloatTensor(X_train).to(device)
y_train = torch.LongTensor(y_train).to(device)
X_test = torch.FloatTensor(X_test).to(device)
y_test = torch.LongTensor(y_test).to(device)

# 모델 학습 함수
def train_model(model, X_train, y_train, epochs=100):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    
    losses = []
    
    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()
        
        losses.append(loss.item())
        
        if (epoch + 1) % 20 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')
    
    return losses

# Sigmoid 모델 학습
print("\n=== Sigmoid 활성화 함수 사용 ===")
model_sigmoid = DeepNetworkSigmoid(2, 64, 5, 2).to(device)
losses_sigmoid = train_model(model_sigmoid, X_train, y_train)

# ReLU 모델 학습
print("\n=== ReLU 활성화 함수 사용 ===")
model_relu = DeepNetworkReLU(2, 64, 5, 2).to(device)
losses_relu = train_model(model_relu, X_train, y_train)

# 결과 시각화
plt.figure(figsize=(10, 5))
plt.plot(losses_sigmoid, label='Sigmoid', alpha=0.7)
plt.plot(losses_relu, label='ReLU', alpha=0.7)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Sigmoid vs ReLU: Training Loss Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 6.1.3. ReLU 그 후... (다양한 ReLU 변형)

In [ ]:
# 다양한 ReLU 변형 함수들
x = torch.linspace(-5, 5, 100)

# ReLU 변형들
relu = F.relu(x)
leaky_relu = F.leaky_relu(x, negative_slope=0.01)
elu = F.elu(x)
selu = F.selu(x)
gelu = F.gelu(x)

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(x.numpy(), relu.numpy(), label='ReLU', linewidth=2)
plt.plot(x.numpy(), leaky_relu.numpy(), label='Leaky ReLU', linewidth=2)
plt.grid(True, alpha=0.3)
plt.xlabel('Input')
plt.ylabel('Output')
plt.title('ReLU Variants (Linear)')
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(x.numpy(), elu.numpy(), label='ELU', linewidth=2, color='purple')
plt.plot(x.numpy(), selu.numpy(), label='SELU', linewidth=2, color='orange')
plt.grid(True, alpha=0.3)
plt.xlabel('Input')
plt.ylabel('Output')
plt.title('ReLU Variants (Exponential)')
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(x.numpy(), gelu.numpy(), label='GELU', linewidth=2, color='green')
plt.plot(x.numpy(), relu.numpy(), label='ReLU', linewidth=2, alpha=0.5)
plt.grid(True, alpha=0.3)
plt.xlabel('Input')
plt.ylabel('Output')
plt.title('GELU (Transformer에서 주로 사용)')
plt.legend()

plt.tight_layout()
plt.show()

print("\n=== 활성화 함수 특징 ===")
print("ReLU: 가장 기본적인 형태, Dying ReLU 문제 있음")
print("Leaky ReLU: 음수 영역에 작은 기울기 추가, Dying ReLU 완화")
print("ELU: 음수 영역에서 지수 함수 사용, 평균이 0에 가까움")
print("SELU: Self-Normalizing, 특정 조건에서 정규화 효과")
print("GELU: Transformer 모델에서 주로 사용, 확률적 특성")

### 6.1.4. 배치 정규화 (Batch Normalization)

In [ ]:
# Batch Normalization을 사용한 모델
class DeepNetworkWithBN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(DeepNetworkWithBN, self).__init__()
        self.layers = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        
        # 첫 번째 레이어
        self.layers.append(nn.Linear(input_size, hidden_size))
        self.batch_norms.append(nn.BatchNorm1d(hidden_size))
        
        # 히든 레이어들
        for _ in range(num_layers - 1):
            self.layers.append(nn.Linear(hidden_size, hidden_size))
            self.batch_norms.append(nn.BatchNorm1d(hidden_size))
        
        # 출력 레이어
        self.output = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        for layer, bn in zip(self.layers, self.batch_norms):
            x = F.relu(bn(layer(x)))
        x = self.output(x)
        return x

# Batch Normalization 없는 모델
class DeepNetworkWithoutBN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(DeepNetworkWithoutBN, self).__init__()
        self.layers = nn.ModuleList()
        
        # 첫 번째 레이어
        self.layers.append(nn.Linear(input_size, hidden_size))
        
        # 히든 레이어들
        for _ in range(num_layers - 1):
            self.layers.append(nn.Linear(hidden_size, hidden_size))
        
        # 출력 레이어
        self.output = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        for layer in self.layers:
            x = F.relu(layer(x))
        x = self.output(x)
        return x

# MNIST 데이터 로드
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, 
                                          download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

test_dataset = torchvision.datasets.MNIST(root='./data', train=False, 
                                         download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

print("데이터셋 로드 완료!")
print(f"학습 데이터: {len(train_dataset)}개")
print(f"테스트 데이터: {len(test_dataset)}개")

### 6.1.5. 배치 정규화 실험 결과 분석

In [ ]:
# 모델 학습 함수 (MNIST용)
def train_mnist_model(model, train_loader, test_loader, epochs=10):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    train_losses = []
    test_accuracies = []
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        
        for images, labels in train_loader:
            images = images.view(-1, 28*28).to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        avg_loss = running_loss / len(train_loader)
        train_losses.append(avg_loss)
        
        # 테스트 정확도 계산
        model.eval()
        correct = 0
        total = 0
        
        with torch.no_grad():
            for images, labels in test_loader:
                images = images.view(-1, 28*28).to(device)
                labels = labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        accuracy = 100 * correct / total
        test_accuracies.append(accuracy)
        
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%')
    
    return train_losses, test_accuracies

# BN 없는 모델 학습
print("\n=== Batch Normalization 없이 학습 ===")
model_without_bn = DeepNetworkWithoutBN(784, 256, 8, 10).to(device)
losses_without_bn, acc_without_bn = train_mnist_model(model_without_bn, train_loader, test_loader)

# BN 있는 모델 학습
print("\n=== Batch Normalization 사용하여 학습 ===")
model_with_bn = DeepNetworkWithBN(784, 256, 8, 10).to(device)
losses_with_bn, acc_with_bn = train_mnist_model(model_with_bn, train_loader, test_loader)

# 결과 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(losses_without_bn, label='Without BN', marker='o')
axes[0].plot(losses_with_bn, label='With BN', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(acc_without_bn, label='Without BN', marker='o')
axes[1].plot(acc_with_bn, label='With BN', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Test Accuracy Comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 6.1.6. 레이어 정규화 (Layer Normalization)

In [ ]:
# Layer Normalization을 사용한 모델
class DeepNetworkWithLN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(DeepNetworkWithLN, self).__init__()
        self.layers = nn.ModuleList()
        self.layer_norms = nn.ModuleList()
        
        # 첫 번째 레이어
        self.layers.append(nn.Linear(input_size, hidden_size))
        self.layer_norms.append(nn.LayerNorm(hidden_size))
        
        # 히든 레이어들
        for _ in range(num_layers - 1):
            self.layers.append(nn.Linear(hidden_size, hidden_size))
            self.layer_norms.append(nn.LayerNorm(hidden_size))
        
        # 출력 레이어
        self.output = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        for layer, ln in zip(self.layers, self.layer_norms):
            x = F.relu(ln(layer(x)))
        x = self.output(x)
        return x

# Batch Normalization vs Layer Normalization 비교
print("\n=== Batch Normalization vs Layer Normalization ===")
print("\nBatch Normalization:")
print("- 배치의 각 특성(feature)에 대해 정규화")
print("- 배치 크기에 의존적")
print("- CNN에서 주로 사용")
print("- 학습/추론 시 다르게 동작 (running mean/var 사용)")

print("\nLayer Normalization:")
print("- 각 샘플의 모든 특성에 대해 정규화")
print("- 배치 크기에 무관")
print("- RNN, Transformer에서 주로 사용")
print("- 학습/추론 시 동일하게 동작")

# Layer Normalization 시각화
batch_size = 4
features = 5

# 예제 데이터
x = torch.randn(batch_size, features)

# Batch Normalization
bn = nn.BatchNorm1d(features)
x_bn = bn(x)

# Layer Normalization  
ln = nn.LayerNorm(features)
x_ln = ln(x)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 원본 데이터
im1 = axes[0].imshow(x.detach().numpy(), cmap='RdBu', aspect='auto')
axes[0].set_title('Original Data')
axes[0].set_xlabel('Features')
axes[0].set_ylabel('Batch')
plt.colorbar(im1, ax=axes[0])

# Batch Normalization
im2 = axes[1].imshow(x_bn.detach().numpy(), cmap='RdBu', aspect='auto')
axes[1].set_title('Batch Normalization\n(각 특성별로 정규화)')
axes[1].set_xlabel('Features')
axes[1].set_ylabel('Batch')
plt.colorbar(im2, ax=axes[1])

# Layer Normalization
im3 = axes[2].imshow(x_ln.detach().numpy(), cmap='RdBu', aspect='auto')
axes[2].set_title('Layer Normalization\n(각 샘플별로 정규화)')
axes[2].set_xlabel('Features')
axes[2].set_ylabel('Batch')
plt.colorbar(im3, ax=axes[2])

plt.tight_layout()
plt.show()

## 6.2. Loss Landscape 문제와 ResNet의 Skip-Connection

In [ ]:
# Residual Block 정의
class ResidualBlock(nn.Module):
    def __init__(self, hidden_size):
        super(ResidualBlock, self).__init__()
        self.fc1 = nn.Linear(hidden_size, hidden_size)
        self.bn1 = nn.BatchNorm1d(hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.bn2 = nn.BatchNorm1d(hidden_size)
    
    def forward(self, x):
        identity = x  # Skip Connection
        
        out = F.relu(self.bn1(self.fc1(x)))
        out = self.bn2(self.fc2(out))
        
        out += identity  # Residual Connection
        out = F.relu(out)
        
        return out

# ResNet 스타일의 깊은 네트워크
class ResNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_blocks, output_size):
        super(ResNet, self).__init__()
        
        self.input_layer = nn.Linear(input_size, hidden_size)
        
        self.residual_blocks = nn.ModuleList(
            [ResidualBlock(hidden_size) for _ in range(num_blocks)]
        )
        
        self.output_layer = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        x = F.relu(self.input_layer(x))
        
        for block in self.residual_blocks:
            x = block(x)
        
        x = self.output_layer(x)
        return x

# Skip Connection이 없는 일반 깊은 네트워크
class PlainDeepNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(PlainDeepNet, self).__init__()
        
        layers = []
        layers.append(nn.Linear(input_size, hidden_size))
        layers.append(nn.BatchNorm1d(hidden_size))
        layers.append(nn.ReLU())
        
        for _ in range(num_layers):
            layers.append(nn.Linear(hidden_size, hidden_size))
            layers.append(nn.BatchNorm1d(hidden_size))
            layers.append(nn.ReLU())
        
        layers.append(nn.Linear(hidden_size, output_size))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

# Skip Connection 효과 시각화
print("\n=== Skip Connection (Residual Connection)의 장점 ===")
print("1. 기울기 소실 문제 완화: 역전파 시 직접적인 경로 제공")
print("2. 깊은 네트워크 학습 가능: 수백 개의 레이어도 학습 가능")
print("3. Identity Mapping: 최악의 경우 입력을 그대로 전달 가능")
print("4. Loss Landscape 완만화: 최적화가 더 쉬워짐")

# ResNet 학습
print("\n=== ResNet (Skip Connection 사용) 학습 ===")
resnet = ResNet(784, 256, 10, 10).to(device)
losses_resnet, acc_resnet = train_mnist_model(resnet, train_loader, test_loader, epochs=5)

# Plain Network 학습
print("\n=== Plain Deep Network (Skip Connection 없음) 학습 ===")
plain_net = PlainDeepNet(784, 256, 10, 10).to(device)
losses_plain, acc_plain = train_mnist_model(plain_net, train_loader, test_loader, epochs=5)

# 결과 비교
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(losses_plain, label='Plain Network', marker='o')
axes[0].plot(losses_resnet, label='ResNet', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss: Skip Connection의 효과')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(acc_plain, label='Plain Network', marker='o')
axes[1].plot(acc_resnet, label='ResNet', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Test Accuracy: Skip Connection의 효과')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6.3. 과적합 (Overfitting)

### 6.3.1. 데이터 증강 (Data Augmentation)

In [ ]:
# 데이터 증강 기법들
transform_with_augmentation = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# 데이터 증강 예시 시각화
sample_image, _ = train_dataset[0]

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Data Augmentation 예시', fontsize=16)

# 원본 이미지
axes[0, 0].imshow(sample_image.squeeze(), cmap='gray')
axes[0, 0].set_title('Original')
axes[0, 0].axis('off')

# 다양한 증강 기법 적용
augmentation_techniques = [
    ('Random Rotation', transforms.RandomRotation(15)),
    ('Random Translate', transforms.RandomAffine(degrees=0, translate=(0.15, 0.15))),
    ('Random Scale', transforms.RandomAffine(degrees=0, scale=(0.8, 1.2))),
    ('Random Shear', transforms.RandomAffine(degrees=0, shear=10)),
]

from PIL import Image
pil_image = Image.fromarray((sample_image.squeeze().numpy() * 255).astype(np.uint8))

for idx, (name, transform) in enumerate(augmentation_techniques, 1):
    augmented = transform(pil_image)
    axes[0, idx].imshow(np.array(augmented), cmap='gray')
    axes[0, idx].set_title(name)
    axes[0, idx].axis('off')

# 여러 번 증강 적용
for idx in range(5):
    augmented = transform_with_augmentation(pil_image)
    axes[1, idx].imshow(augmented.squeeze(), cmap='gray')
    axes[1, idx].set_title(f'Combined Aug {idx+1}')
    axes[1, idx].axis('off')

plt.tight_layout()
plt.show()

print("\n=== 데이터 증강의 효과 ===")
print("1. 학습 데이터의 다양성 증가")
print("2. 모델의 일반화 성능 향상")
print("3. 과적합 방지")
print("4. 적은 데이터로도 robust한 모델 학습 가능")

### 6.3.2. Dropout

In [ ]:
# Dropout을 사용한 모델
class NetworkWithDropout(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_rate, output_size):
        super(NetworkWithDropout, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, hidden_size)
        self.fc4 = nn.Linear(hidden_size, output_size)
        
        self.dropout = nn.Dropout(dropout_rate)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        
        x = F.relu(self.fc3(x))
        x = self.dropout(x)
        
        x = self.fc4(x)
        return x

# Dropout 시각화
dropout_rates = [0.0, 0.2, 0.5, 0.8]
x_sample = torch.randn(1, 100)

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
fig.suptitle('Dropout 효과 시각화', fontsize=16)

for idx, rate in enumerate(dropout_rates):
    dropout = nn.Dropout(rate)
    dropout.train()  # 학습 모드로 설정
    x_dropped = dropout(x_sample)
    
    axes[idx].bar(range(100), x_dropped[0].detach().numpy())
    axes[idx].set_title(f'Dropout Rate: {rate}')
    axes[idx].set_xlabel('Neuron Index')
    axes[idx].set_ylabel('Activation')
    axes[idx].set_ylim([-3, 3])

plt.tight_layout()
plt.show()

print("\n=== Dropout의 원리와 효과 ===")
print("1. 학습 시 무작위로 뉴런을 비활성화 (확률 p)")
print("2. 앙상블 효과: 여러 서브 네트워크를 동시에 학습")
print("3. Co-adaptation 방지: 특정 뉴런에 대한 과도한 의존 방지")
print("4. 추론 시에는 모든 뉴런 사용 (출력에 (1-p) 곱함)")
print("5. 과적합 방지 및 일반화 성능 향상")

### 6.3.3. 오토인코더 (Autoencoder)에 Dropout 적용

In [ ]:
# Autoencoder 정의
class Autoencoder(nn.Module):
    def __init__(self, input_size, hidden_size, latent_size, dropout_rate=0.0):
        super(Autoencoder, self).__init__()
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_size, latent_size),
            nn.ReLU()
        )
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_size, input_size),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

# Autoencoder 학습 함수
def train_autoencoder(model, train_loader, epochs=5):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    losses = []
    
    for epoch in range(epochs):
        running_loss = 0.0
        for images, _ in train_loader:
            images = images.view(-1, 28*28).to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, images)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        avg_loss = running_loss / len(train_loader)
        losses.append(avg_loss)
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}')
    
    return losses

# Dropout 없는 Autoencoder
print("\n=== Dropout 없는 Autoencoder 학습 ===")
autoencoder_no_dropout = Autoencoder(784, 256, 64, dropout_rate=0.0).to(device)
losses_no_dropout = train_autoencoder(autoencoder_no_dropout, train_loader)

# Dropout 있는 Autoencoder
print("\n=== Dropout 있는 Autoencoder 학습 ===")
autoencoder_with_dropout = Autoencoder(784, 256, 64, dropout_rate=0.3).to(device)
losses_with_dropout = train_autoencoder(autoencoder_with_dropout, train_loader)

# 재구성 결과 비교
autoencoder_no_dropout.eval()
autoencoder_with_dropout.eval()

with torch.no_grad():
    test_images, _ = next(iter(test_loader))
    test_images = test_images.to(device)
    test_images_flat = test_images.view(-1, 28*28)
    
    reconstructed_no_dropout = autoencoder_no_dropout(test_images_flat)
    reconstructed_with_dropout = autoencoder_with_dropout(test_images_flat)

# 시각화
fig, axes = plt.subplots(3, 10, figsize=(20, 6))
fig.suptitle('Autoencoder 재구성 결과: Dropout 효과', fontsize=16)

for i in range(10):
    # 원본
    axes[0, i].imshow(test_images[i].cpu().squeeze(), cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_ylabel('Original', fontsize=12)
    
    # Dropout 없음
    axes[1, i].imshow(reconstructed_no_dropout[i].cpu().view(28, 28), cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_ylabel('No Dropout', fontsize=12)
    
    # Dropout 있음
    axes[2, i].imshow(reconstructed_with_dropout[i].cpu().view(28, 28), cmap='gray')
    axes[2, i].axis('off')
    if i == 0:
        axes[2, i].set_ylabel('With Dropout', fontsize=12)

plt.tight_layout()
plt.show()

### 6.3.4. Regularization

In [ ]:
# L2 Regularization (Weight Decay)를 사용한 모델
class SimpleNetwork(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleNetwork, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# 다양한 Weight Decay 값으로 학습
def train_with_regularization(weight_decay, epochs=10):
    model = SimpleNetwork(784, 256, 10).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=weight_decay)
    
    train_losses = []
    test_accuracies = []
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        
        for images, labels in train_loader:
            images = images.view(-1, 28*28).to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        train_losses.append(running_loss / len(train_loader))
        
        # 테스트 정확도
        model.eval()
        correct = 0
        total = 0
        
        with torch.no_grad():
            for images, labels in test_loader:
                images = images.view(-1, 28*28).to(device)
                labels = labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        test_accuracies.append(100 * correct / total)
    
    return train_losses, test_accuracies, model

# 다양한 weight decay 값으로 실험
weight_decays = [0.0, 0.001, 0.01, 0.1]
results = {}

for wd in weight_decays:
    print(f"\n=== Weight Decay = {wd} ===")
    losses, accs, model = train_with_regularization(wd, epochs=10)
    results[wd] = {'losses': losses, 'accuracies': accs, 'model': model}

# 결과 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for wd in weight_decays:
    axes[0].plot(results[wd]['losses'], label=f'WD={wd}', marker='o')
    axes[1].plot(results[wd]['accuracies'], label=f'WD={wd}', marker='o')

axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training Loss')
axes[0].set_title('Weight Decay의 효과 (Loss)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Test Accuracy (%)')
axes[1].set_title('Weight Decay의 효과 (Accuracy)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 6.3.5. Regularization과 MAP (Maximum A Posteriori)

In [ ]:
# MAP와 Regularization의 관계 설명
print("\n=== Regularization과 MAP의 관계 ===")
print("\n1. Maximum Likelihood Estimation (MLE):")
print("   θ_MLE = argmax P(D|θ)")
print("   -> 데이터에만 의존, 과적합 가능성")

print("\n2. Maximum A Posteriori (MAP):")
print("   θ_MAP = argmax P(θ|D) = argmax P(D|θ) * P(θ)")
print("   -> 사전 확률(prior) P(θ) 고려")

print("\n3. L2 Regularization = Gaussian Prior:")
print("   P(θ) ~ exp(-λ||θ||²)")
print("   Loss = -log P(D|θ) + λ||θ||²")
print("   -> Weight Decay와 동일")

print("\n4. L1 Regularization = Laplace Prior:")
print("   P(θ) ~ exp(-λ||θ||₁)")
print("   Loss = -log P(D|θ) + λ||θ||₁")
print("   -> Sparsity 유도 (많은 가중치가 0이 됨)")

# L1 vs L2 Regularization 시각화
theta = np.linspace(-3, 3, 100)

# L2 prior (Gaussian)
l2_prior = np.exp(-0.5 * theta**2)

# L1 prior (Laplace)
l1_prior = np.exp(-np.abs(theta))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Prior 분포
axes[0].plot(theta, l2_prior, label='L2 (Gaussian)', linewidth=2)
axes[0].plot(theta, l1_prior, label='L1 (Laplace)', linewidth=2)
axes[0].set_xlabel('Parameter θ')
axes[0].set_ylabel('Prior P(θ)')
axes[0].set_title('Prior Distributions')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Regularization Term
l2_reg = theta**2
l1_reg = np.abs(theta)

axes[1].plot(theta, l2_reg, label='L2 Regularization', linewidth=2)
axes[1].plot(theta, l1_reg, label='L1 Regularization', linewidth=2)
axes[1].set_xlabel('Parameter θ')
axes[1].set_ylabel('Regularization Term')
axes[1].set_title('Regularization Terms')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Gradient
l2_grad = 2 * theta
l1_grad = np.sign(theta)

axes[2].plot(theta, l2_grad, label='L2 Gradient', linewidth=2)
axes[2].plot(theta, l1_grad, label='L1 Gradient', linewidth=2)
axes[2].set_xlabel('Parameter θ')
axes[2].set_ylabel('Gradient')
axes[2].set_title('Gradients')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 6.3.6. L2-Regularization vs L1-Regularization 실험 결과 분석

In [ ]:
# L1 Regularization을 수동으로 구현
def train_with_l1_regularization(l1_lambda, epochs=10):
    model = SimpleNetwork(784, 256, 10).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    train_losses = []
    test_accuracies = []
    sparsity_ratios = []  # 0에 가까운 가중치 비율
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        
        for images, labels in train_loader:
            images = images.view(-1, 28*28).to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # L1 Regularization 추가
            l1_reg = torch.tensor(0., requires_grad=True).to(device)
            for param in model.parameters():
                l1_reg = l1_reg + torch.norm(param, 1)
            
            total_loss = loss + l1_lambda * l1_reg
            total_loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        train_losses.append(running_loss / len(train_loader))
        
        # Sparsity 계산 (가중치가 0.01 이하인 비율)
        total_params = 0
        sparse_params = 0
        for param in model.parameters():
            total_params += param.numel()
            sparse_params += (torch.abs(param) < 0.01).sum().item()
        sparsity_ratios.append(100 * sparse_params / total_params)
        
        # 테스트 정확도
        model.eval()
        correct = 0
        total = 0
        
        with torch.no_grad():
            for images, labels in test_loader:
                images = images.view(-1, 28*28).to(device)
                labels = labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        test_accuracies.append(100 * correct / total)
    
    return train_losses, test_accuracies, sparsity_ratios, model

# L2 vs L1 비교
print("\n=== L2 Regularization (Weight Decay=0.01) ===")
l2_losses, l2_accs, l2_model = train_with_regularization(weight_decay=0.01, epochs=10)

print("\n=== L1 Regularization (Lambda=0.0001) ===")
l1_losses, l1_accs, l1_sparsity, l1_model = train_with_l1_regularization(l1_lambda=0.0001, epochs=10)

# 가중치 분포 비교
l2_weights = []
l1_weights = []

for param in l2_model.parameters():
    l2_weights.extend(param.detach().cpu().numpy().flatten())

for param in l1_model.parameters():
    l1_weights.extend(param.detach().cpu().numpy().flatten())

# 시각화
fig = plt.figure(figsize=(18, 5))

# 학습 곡선
ax1 = plt.subplot(1, 3, 1)
ax1.plot(l2_losses, label='L2 Reg', marker='o')
ax1.plot(l1_losses, label='L1 Reg', marker='s')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Training Loss')
ax1.set_title('L1 vs L2: Training Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 가중치 분포
ax2 = plt.subplot(1, 3, 2)
ax2.hist(l2_weights, bins=50, alpha=0.5, label='L2 Reg', density=True)
ax2.hist(l1_weights, bins=50, alpha=0.5, label='L1 Reg', density=True)
ax2.set_xlabel('Weight Value')
ax2.set_ylabel('Density')
ax2.set_title('Weight Distribution')
ax2.set_xlim([-0.5, 0.5])
ax2.legend()
ax2.grid(True, alpha=0.3)

# Sparsity
ax3 = plt.subplot(1, 3, 3)
ax3.plot(l1_sparsity, label='L1 Sparsity', marker='o', color='red')
ax3.set_xlabel('Epoch')
ax3.set_ylabel('Sparsity (%)')
ax3.set_title('L1 Regularization: Weight Sparsity')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== L1 vs L2 Regularization 비교 ===")
print(f"L2 Regularization:")
print(f"  - 최종 정확도: {l2_accs[-1]:.2f}%")
print(f"  - 가중치 0에 가까운 비율: {100 * np.sum(np.abs(l2_weights) < 0.01) / len(l2_weights):.2f}%")
print(f"\nL1 Regularization:")
print(f"  - 최종 정확도: {l1_accs[-1]:.2f}%")
print(f"  - 가중치 0에 가까운 비율 (Sparsity): {l1_sparsity[-1]:.2f}%")
print(f"\n결론: L1은 더 많은 가중치를 0으로 만들어 Feature Selection 효과")

## 종합 정리

In [ ]:
print("\n" + "="*70)
print("Chapter 6: 깊은 인공 신경망의 고질적 문제와 해결 방안 - 종합 정리")
print("="*70)

print("\n[6.1] 기울기 소실 문제 해결:")
print("  1. ReLU 활성화 함수: Sigmoid 대비 기울기 소실 완화")
print("  2. ReLU 변형: Leaky ReLU, ELU, SELU, GELU 등")
print("  3. Batch Normalization: 내부 공변량 이동 감소, 학습 안정화")
print("  4. Layer Normalization: 배치 크기에 독립적, RNN/Transformer에 적합")

print("\n[6.2] Loss Landscape 문제:")
print("  1. Skip Connection (ResNet): 기울기 직접 전파 경로 제공")
print("  2. Identity Mapping: 최악의 경우에도 입력 보존")
print("  3. Loss Landscape 완만화: 최적화 용이")

print("\n[6.3] 과적합 문제 해결:")
print("  1. Data Augmentation: 학습 데이터 다양성 증가")
print("  2. Dropout: 앙상블 효과, Co-adaptation 방지")
print("  3. L2 Regularization (Weight Decay): Gaussian Prior, 가중치 크기 제한")
print("  4. L1 Regularization: Laplace Prior, Sparsity 유도")
print("  5. MAP: 사전 확률을 고려한 최적화")

print("\n[실무 적용 가이드]")
print("  • 활성화 함수: ReLU 또는 GELU (Transformer)")
print("  • 정규화: CNN은 Batch Norm, RNN/Transformer는 Layer Norm")
print("  • 깊은 네트워크: Skip Connection 필수")
print("  • 과적합 방지: Dropout + Data Augmentation + Weight Decay 조합")
print("  • Hyperparameter: 데이터와 모델에 따라 실험적으로 결정")
print("\n" + "="*70)